In [1]:
import pandas as pd

print("Python 3.12 is working!")
print(pd.__version__)

Python 3.12 is working!
3.0.5


In [2]:
import pandas as pd

In [3]:
# LOADING OF DATASET

df = pd.read_csv("PS_20174392719_1491204439457_log.csv")

print("Dataset loaded successfully!")
print(df.head())
print(df.shape)

Dataset loaded successfully!
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
(6362620, 11)


In [4]:
# TO CHECK FOR MISSING VALUES

print("MIssing Values:")
print(df.isnull().sum())

MIssing Values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [5]:
# UNDERSTANDING DATA TYPES

print(df.dtypes)

step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


In [6]:
# CHECKING FOR DUPLICATE TRANSACTIONS

print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [7]:
# CHECK THE FRAUD DISTRIBUTION

print("Fraud Distribution:")
print(df["isFraud"].value_counts())

print("\nFraud Percentage:")
print(df["isFraud"].value_counts(normalize=True) * 100)

Fraud Distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud Percentage:
isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


In [8]:
# TRANSACTION TYPES ASSOCIATED WITH FRAUD


print("Fraud by Transaction Type:")
print(pd.crosstab(df["type"], df["isFraud"]))

print("\nFraud Rate by Transaction Type:")
print(
    pd.crosstab(df["type"], df["isFraud"], normalize="index") * 100
)

Fraud by Transaction Type:
isFraud         0     1
type                   
CASH_IN   1399284     0
CASH_OUT  2233384  4116
DEBIT       41432     0
PAYMENT   2151495     0
TRANSFER   528812  4097

Fraud Rate by Transaction Type:
isFraud            0         1
type                          
CASH_IN   100.000000  0.000000
CASH_OUT   99.816045  0.183955
DEBIT     100.000000  0.000000
PAYMENT   100.000000  0.000000
TRANSFER   99.231201  0.768799


In [9]:
# WHAT TRANSACTION AMOUNTS ARE RISKY?


print("Average Transaction Amount:")
print(df.groupby("isFraud")["amount"].mean())

print("\nMedian Transaction Amount:")
print(df.groupby("isFraud")["amount"].median())

print("\nMaximum Transaction Amount:")
print(df.groupby("isFraud")["amount"].max())

Average Transaction Amount:
isFraud
0    1.781970e+05
1    1.467967e+06
Name: amount, dtype: float64

Median Transaction Amount:
isFraud
0     74684.72
1    441423.44
Name: amount, dtype: float64

Maximum Transaction Amount:
isFraud
0    92445516.64
1    10000000.00
Name: amount, dtype: float64


In [10]:
# IDENTIFYING SUSPCIOUS TRANSACTION PATTERNS


print("Average Amount by Transaction Type and Fraud Status:")

print(
    df.groupby(["type", "isFraud"])["amount"]
      .mean()
      .round(2)
)

Average Amount by Transaction Type and Fraud Status:
type      isFraud
CASH_IN   0           168920.24
CASH_OUT  0           173917.16
          1          1455102.59
DEBIT     0             5483.67
PAYMENT   0            13057.60
TRANSFER  0           906229.01
          1          1480891.67
Name: amount, dtype: float64


Fraud is concentrated in CASH_OUT and TRANSFER transactions, and fraudulent transactions tend to involve substantially larger amounts.

In [11]:
# CREATE A COLUMN CALLED SENDER BALANCE CHANGE TO FIND OUT IF THE SENDER BALANCE CHANGES



print("Fraud by Sender Balance Change:")

df["sender_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

print(
    df.groupby("isFraud")["sender_balance_change"]
      .mean()
      .round(2)
)

Fraud by Sender Balance Change:
isFraud
0     -23141.52
1    1457274.97
Name: sender_balance_change, dtype: float64


In [12]:
# TO CHECK BALANCE CONSISTENCY


df["balance_error"] = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
)

print("Average Balance Error:")
print(
    df.groupby("isFraud")["balance_error"]
      .mean()
      .round(2)
)

Average Balance Error:
isFraud
0   -201338.56
1    -10692.33
Name: balance_error, dtype: float64


In [13]:
# RECIPIENT'S BALANCE

print("Average Recipient Balance Change:")

df["recipient_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

print(
    df.groupby("isFraud")["recipient_balance_change"]
      .mean()
      .round(2)
)

Average Recipient Balance Change:
isFraud
0    123504.81
1    735458.00
Name: recipient_balance_change, dtype: float64


Fraudulent transactions in this dataset tend to involve substantially larger recipient-side balance changes than legitimate transactions.

In [ ]:
# FRAUD RATE BY TRANSACTION TYPE BAR CHART


import matplotlib.pyplot as plt

fraud_rate = (
    df.groupby("type")["isFraud"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print("Fraud Rate by Transaction Type:")
print(fraud_rate.round(4))

plt.figure(figsize=(8, 5))
fraud_rate.plot(kind="bar")

plt.title("Fraud Rate by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# TRANSACTION TYPE ANALYSIS

transaction_summary = df.groupby("type").agg(
    transaction_count=("type", "size"),
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean")
)

transaction_summary["percentage_of_transactions"] = (
    transaction_summary["transaction_count"]
    / len(df) * 100
)

transaction_summary = transaction_summary.sort_values(
    "transaction_count",
    ascending=False
)

print("TRANSACTION TYPE SUMMARY:")
print(transaction_summary.round(2))

print("\nMOST FREQUENT TRANSACTION TYPE:")
print(transaction_summary["transaction_count"].idxmax())

print("\nLEAST FREQUENT TRANSACTION TYPE:")
print(transaction_summary["transaction_count"].idxmin())

print("\nHIGHEST TOTAL TRANSACTION AMOUNT:")
print(transaction_summary["total_amount"].idxmax())

print("\nLARGEST AVERAGE TRANSACTION VALUE:")
print(transaction_summary["average_amount"].idxmax())

In [ ]:
# TRANSACTION TRENDS OVER TIME

transaction_trend = df.groupby("step").size()

print("Transaction Volume Over Time:")
print(transaction_trend.head(20))

print("\nHighest Transaction Volume:")
print("Step:", transaction_trend.idxmax())
print("Transactions:", transaction_trend.max())

print("\nLowest Transaction Volume:")
print("Step:", transaction_trend.idxmin())
print("Transactions:", transaction_trend.min())

In [ ]:
# TRANSACTION VOLUME TIME SERIES

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

transaction_trend.plot()

plt.title("Transaction Volume Over Time")
plt.xlabel("Step")
plt.ylabel("Number of Transactions")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# USER BEHAVIOUR AND SUSPICIOUS PATTERNS

# Sender balance change
df["sender_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

# Recipient balance change
df["recipient_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

# Check how balances behave for normal vs fraudulent transactions
print("AVERAGE SENDER BALANCE CHANGE:")
print(
    df.groupby("isFraud")["sender_balance_change"]
    .mean()
    .round(2)
)

print("\nAVERAGE RECIPIENT BALANCE CHANGE:")
print(
    df.groupby("isFraud")["recipient_balance_change"]
    .mean()
    .round(2)
)

print("\nAVERAGE OLD SENDER BALANCE:")
print(
    df.groupby("isFraud")["oldbalanceOrg"]
    .mean()
    .round(2)
)

print("\nAVERAGE NEW SENDER BALANCE:")
print(
    df.groupby("isFraud")["newbalanceOrig"]
    .mean()
    .round(2)
)

In [ ]:
# ACCOUNT EMPTYING PATTERN

account_emptying = df[
    (df["oldbalanceOrg"] > 0) &
    (df["newbalanceOrig"] == 0)
]

print("ACCOUNT-EMPTYING TRANSACTIONS:")
print(account_emptying["isFraud"].value_counts())

print("\nACCOUNT-EMPTYING FRAUD RATE:")
print(
    account_emptying["isFraud"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
)

In [ ]:
# HIGH-VALUE TRANSACTION BEHAVIOUR

high_value_threshold = df["amount"].quantile(0.99)

high_value = df[df["amount"] >= high_value_threshold]

print("99TH PERCENTILE TRANSACTION AMOUNT:")
print(round(high_value_threshold, 2))

print("\nHIGH-VALUE TRANSACTIONS:")
print(high_value["isFraud"].value_counts())

print("\nFRAUD RATE AMONG HIGH-VALUE TRANSACTIONS:")
print(
    high_value["isFraud"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
)

In [ ]:
# FRAUD FLAGGING SYSTEM EFFECTIVENESS

print("CONFUSION MATRIX:")
print(pd.crosstab(
    df["isFraud"],
    df["isFlaggedFraud"],
    rownames=["Actual Fraud"],
    colnames=["Flagged Fraud"]
))

# True Positives
true_positives = ((df["isFraud"] == 1) & (df["isFlaggedFraud"] == 1)).sum()

# False Negatives
false_negatives = ((df["isFraud"] == 1) & (df["isFlaggedFraud"] == 0)).sum()

# False Positives
false_positives = ((df["isFraud"] == 0) & (df["isFlaggedFraud"] == 1)).sum()

# True Negatives
true_negatives = ((df["isFraud"] == 0) & (df["isFlaggedFraud"] == 0)).sum()

print("\nTRUE POSITIVES:", true_positives)
print("FALSE NEGATIVES:", false_negatives)
print("FALSE POSITIVES:", false_positives)
print("TRUE NEGATIVES:", true_negatives)

print("\nFLAGGING RATE AMONG ACTUAL FRAUD:")
print(
    round(
        true_positives / (true_positives + false_negatives) * 100,
        4
    ),
    "%"
)

print("\nMISSED FRAUD RATE:")
print(
    round(
        false_negatives / (true_positives + false_negatives) * 100,
        4
    ),
    "%"
)

The existing isFlaggedFraud mechanism has extremely low fraud-detection coverage. It correctly identifies only 0.1948% of actual fraudulent transactions while missing 99.8052%. Although it produces no false positives in this dataset, its extremely high false-negative rate indicates that the current flagging mechanism is insufficient for detecting fraudulent activity.

In [ ]:
# TRANSACTION TREND OVER TIME

transaction_trend = df.groupby("step").size()

print("TRANSACTION VOLUME OVER TIME:")
print(transaction_trend.head(20))

print("\nTOTAL STEPS:", df["step"].nunique())
print("FIRST STEP:", df["step"].min())
print("LAST STEP:", df["step"].max())

In [ ]:
# FRAUD TREND OVER TIME

fraud_trend = df.groupby("step")["isFraud"].sum()

print("FRAUD TRANSACTIONS OVER TIME:")
print(fraud_trend.head(20))

print("\nSTEP WITH HIGHEST FRAUD:")
print(fraud_trend.idxmax())

print("HIGHEST NUMBER OF FRAUD TRANSACTIONS:")
print(fraud_trend.max())

CONCLUSION

Transaction trends: The dataset contains 6.36 million transactions across 743 time steps. Transaction volume increases substantially from the beginning of the observation period, although activity fluctuates across individual steps. Fraudulent transactions are also distributed unevenly over time, with the highest number of fraudulent transactions occurring at step 212, where 40 fraud cases were recorded. This suggests that transaction activity and fraud risk can vary over time, making temporal monitoring an important component of fraud detection.

In [ ]:
# TRANSACTION VOLUME OVER TIME - LINE CHART

import matplotlib.pyplot as plt

transaction_trend = df.groupby("step").size()

plt.figure(figsize=(12, 6))
plt.plot(transaction_trend.index, transaction_trend.values)

plt.title("Transaction Volume Over Time")
plt.xlabel("Step")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()

In [ ]:
# SAVE CLEANED DATASET

df.to_csv("sidonpay_cleaned_dataset.csv", index=False)

print("Cleaned dataset saved successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])